In [1]:
from Data_query.trino_config import *
import json
import numpy as np
import matplotlib.pyplot as plt
from visualisation import *
import pytz

In [22]:
stop_trino()


Trino service stopping triggered.


In [27]:
big_workers = 0
workers = 1
num_workers = max(workers, big_workers)
ensure_trino_running(worker_desired_count = workers, big_worker_desired_count=big_workers)
sleep(120)


Trino service is not running. Starting the service...
Trino service triggered.
Service trino-service is now stable.


In [4]:
iceberg_sql(("""select * 
             from conformance_antiisland 
             where site_id = 625794481 and year=2024 and month=10
             limit 5""")).head()

,year,month,day,day_night,site_id,nonconformance_antiisland_sum,nonconformance_antiisland_count,total_count,v_threshold
0,2024,10,18,day,625794481,159.079047,30,30,265.0
1,2024,10,19,day,625794481,1098.305392,134,136,265.0
2,2024,10,20,day,625794481,1116.523626,133,135,265.0
3,2024,10,21,day,625794481,1229.447585,132,137,265.0
4,2024,10,21,night,625794481,0.018873,1,3,265.0


In [8]:
iceberg_exec("DROP TABLE IF EXISTS conformance_antiisland")
iceberg_exec("""CREATE TABLE conformance_antiisland (
                year INT,
                month INT,
                day INT,
                day_night VARCHAR,
                site_id BIGINT,
                nonconformance_antiisland_sum DOUBLE,
                nonconformance_antiisland_count BIGINT,
                total_count BIGINT,
                v_threshold DOUBLE
            )
    """)

Executed
Executed


In [ ]:
def run_func(args):
    year, month, v_threshold, split_cons = args
    iceberg_exec(f"""insert into conformance_antiisland
                        with data as (
                            select site_id, t_stamp, sum(cast(power*circuit_polarity as decimal(18, 6)))/1000 as P_kW, 
                                    max(cast(voltage as decimal(18, 6))) as V, ac_capacity_kw
                        from 
                        (select circuit_id, t_stamp, power, energy_reactive, voltage 
                        from ts where year={year} and month={month} and is_pv = true and voltage > 0 and voltage < 350 
                        and {split_cons} 
                        ) as ts1
                        inner join (select site_id, circuit_id, circuit_polarity,  ac_capacity_kw from meta_up23c 
                        ) as m on ts1.circuit_id = m.circuit_id
                        group by site_id, t_stamp,  ac_capacity_kw),
                        data1 as (
                        select site_id, t_stamp, V, P_kW, ac_capacity_kw,
                        lag(t_stamp) over (partition by site_id order by t_stamp) as prev_t_stamp,
                        lag(t_stamp, 2) over (partition by site_id order by t_stamp) as prev_t_stamp_2,
                        lag(V) over (partition by site_id order by t_stamp) as prev_V,
                        lag(V, 2) over (partition by site_id order by t_stamp) as prev_V_2,
                        lag(P_kW) over (partition by site_id order by t_stamp) as prev_P_kW,
                        lag(P_kW, 2) over (partition by site_id order by t_stamp) as prev_P_kW_2
                        from data
                        ),
                        data2 as (
                        select site_id, t_stamp, V, P_kW, ac_capacity_kw, day(t_stamp) as day,
                        case when (hour(t_stamp) >= 20 or hour(t_stamp) <= 7) then 'day' else 'night' end as day_night,
                        case when P_kW >= 0.04 * ac_capacity_kw then P_kW - 0.04 * ac_capacity_kw else 0 end as nonconformance_sum
                        from data1
                        where V >= {v_threshold} and prev_V >= {v_threshold} and (P_kW > 0.04 * ac_capacity_kw or prev_P_kW > 0.04 * ac_capacity_kw 
                        or prev_P_kW_2 > 0.04 * ac_capacity_kw)
                        )
                        select {year} as year, {month} as month, day, day_night, site_id,  
                        sum(nonconformance_sum) as nonconformance_antiisland_sum, 
                        sum(case when nonconformance_sum > 0 then 1 else 0 end) as nonconformance_antiisland_count,
                        count(nonconformance_sum) as total_count,
                        {v_threshold} as v_threshold
                        from data2
                        group by site_id, day, day_night
                        """)
                        # 
    sleep(20)
    print(f"Completed year={year}, month={month}, v_threshold={v_threshold}, {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}")
    return None

tasks = [(year, month, v_threshold, split_cons) for year in (2024, 2025) for month in range(1, 13) for v_threshold in range(260, 266)  
         for split_cons in ['system.bucket(postcode, 16) <= 3', '(system.bucket(postcode, 16) > 3 and system.bucket(postcode, 16) <= 7)', 
                            '(system.bucket(postcode, 16) > 7 and system.bucket(postcode, 16) <= 11)', 'system.bucket(postcode, 16) > 11'] ]
        #  for split_cons in [f'system.bucket(postcode, 16) = {i}' for i in range(16)] ]

        #  for split_cons in ['system.bucket(postcode, 16) <= 1'] ]
         
df = trino_parallel(run_func, tasks, num_workers=num_workers)

Executed
Executed
Executed
Executed
Executed
Executed
Completed year=2024, month=1, v_threshold=261, bucket <= 3
Completed year=2024, month=1, v_threshold=260, (bucket > 3 and bucket <= 7)
Completed year=2024, month=1, v_threshold=260, bucket <= 3
Completed year=2024, month=1, v_threshold=260, bucket > 11
Completed year=2024, month=1, v_threshold=260, (bucket > 7 and bucket <= 11)
Completed year=2024, month=1, v_threshold=261, (bucket > 3 and bucket <= 7)
Executed
Executed
Executed
Executed
Executed
Executed
Completed year=2024, month=1, v_threshold=262, bucket <= 3
Completed year=2024, month=1, v_threshold=261, bucket > 11
Completed year=2024, month=1, v_threshold=262, (bucket > 3 and bucket <= 7)
Completed year=2024, month=1, v_threshold=262, bucket > 11
Completed year=2024, month=1, v_threshold=261, (bucket > 7 and bucket <= 11)
Completed year=2024, month=1, v_threshold=262, (bucket > 7 and bucket <= 11)
Executed
Executed
Executed
Executed
Executed
Executed
Completed year=2024, mont

In [21]:
def run_func(args):
    year, month, v_threshold, split_cons = args
    df =iceberg_sql(f"""with data as (
                            select site_id, t_stamp, sum(cast(power*circuit_polarity as decimal(18, 6)))/1000 as P_kW, 
                                    max(cast(voltage as decimal(18, 6))) as V, ac_capacity_kw
                        from 
                        (select circuit_id, t_stamp, power, energy_reactive, voltage 
                        from ts where year={year} and month={month} and is_pv = true and voltage > 0 and voltage < 350 
                        and {split_cons} 
                        ) as ts1
                        inner join (select site_id, circuit_id, circuit_polarity,  ac_capacity_kw from meta_up23c 
                        ) as m on ts1.circuit_id = m.circuit_id
                        group by site_id, t_stamp,  ac_capacity_kw),
                        data1 as (
                        select site_id, t_stamp, V, P_kW, ac_capacity_kw,
                        lag(t_stamp) over (partition by site_id order by t_stamp) as prev_t_stamp,
                        lag(t_stamp, 2) over (partition by site_id order by t_stamp) as prev_t_stamp_2,
                        lag(V) over (partition by site_id order by t_stamp) as prev_V,
                        lag(V, 2) over (partition by site_id order by t_stamp) as prev_V_2,
                        lag(P_kW) over (partition by site_id order by t_stamp) as prev_P_kW,
                        lag(P_kW, 2) over (partition by site_id order by t_stamp) as prev_P_kW_2
                        from data
                        ),
                        data2 as (
                        select site_id, t_stamp, V, P_kW, ac_capacity_kw, day(t_stamp) as day,
                        case when (hour(t_stamp) >= 20 or hour(t_stamp) <= 7) then 'day' else 'night' end as day_night,
                        case when P_kW >= 0.04 * ac_capacity_kw then P_kW - 0.04 * ac_capacity_kw else 0 end as nonconformance_sum
                        from data1
                        where V >= {v_threshold} and prev_V >= {v_threshold} and (prev_P_kW > 0.04 * ac_capacity_kw or P_kW > 0.04 * ac_capacity_kw 
                        or prev_P_kW_2 > 0.04 * ac_capacity_kw)
                        )
                        select {year} as year, {month} as month, day, day_night, site_id,  
                        sum(nonconformance_sum) as nonconformance_antiisland_sum, 
                        sum(case when nonconformance_sum > 0 then 1 else 0 end) as nonconformance_antiisland_count,
                        count(nonconformance_sum) as total_count,
                        {v_threshold} as v_threshold
                        from data2
                        group by site_id, day, day_night
                        """)
                        # 
    sleep(20)
    print(f"Completed year={year}, month={month}, v_threshold={v_threshold}, {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}")
    return df

tasks = [(year, month, v_threshold, split_cons) for year in (2024, 2025) for month in range(1, 13) for v_threshold in (260, 263)  
         for split_cons in ['system.bucket(postcode, 16) <= 3', '(system.bucket(postcode, 16) > 3 and system.bucket(postcode, 16) <= 7)', 
                            '(system.bucket(postcode, 16) > 7 and system.bucket(postcode, 16) <= 11)', 'system.bucket(postcode, 16) > 11'] ]
        #  for split_cons in [f'system.bucket(postcode, 16) = {i}' for i in range(16)] ]

        #  for split_cons in ['system.bucket(postcode, 16) <= 1'] ]
         
df = trino_parallel(run_func, tasks, num_workers=num_workers)

Completed year=2024, month=1, v_threshold=263, bucket > 11
Completed year=2024, month=1, v_threshold=260, (bucket > 3 and bucket <= 7)
Completed year=2024, month=1, v_threshold=260, bucket <= 3
Completed year=2024, month=1, v_threshold=263, (bucket > 3 and bucket <= 7)
Completed year=2024, month=1, v_threshold=263, bucket <= 3
Completed year=2024, month=1, v_threshold=263, (bucket > 7 and bucket <= 11)
Completed year=2024, month=1, v_threshold=260, (bucket > 7 and bucket <= 11)
Completed year=2024, month=1, v_threshold=260, bucket > 11
Completed year=2024, month=2, v_threshold=260, bucket <= 3
Completed year=2024, month=2, v_threshold=260, (bucket > 3 and bucket <= 7)
Completed year=2024, month=2, v_threshold=260, bucket > 11
Completed year=2024, month=2, v_threshold=263, bucket <= 3
Completed year=2024, month=2, v_threshold=263, (bucket > 3 and bucket <= 7)
Completed year=2024, month=2, v_threshold=263, (bucket > 7 and bucket <= 11)
Completed year=2024, month=2, v_threshold=263, bucke

In [40]:
iceberg_sql("""select count(distinct site_id) 
            from conformance_antiisland 
            where v_threshold = 261
            """)

,_col0
0,142


In [30]:
iceberg_sql("""select count(distinct site_id) 
            from conformance_antiisland 
            where v_threshold = 260
            """)

,_col0
0,202


In [25]:
df.groupby(['v_threshold']).count().reset_index()

,v_threshold,year,month,day,day_night,site_id,nonconformance_antiisland_sum,nonconformance_antiisland_count,total_count
0,260,4032,4032,4032,4032,4032,4032,4032,4032
1,263,918,918,918,918,918,918,918,918


In [25]:
def run_func(args):
    year, month, split_cons = args
    df = iceberg_sql(f"""
                        with data as (
                            select site_id, t_stamp, sum(cast(power*circuit_polarity as decimal(18, 6)))/1000 as P_kW, 
                                    max(cast(voltage as decimal(18, 6))) as V, ac_capacity_kw
                        from 
                        (select circuit_id, t_stamp, power, energy_reactive, voltage 
                        from ts where year={year} and month={month} and is_pv = true and voltage > 0 and voltage < 300 
                        and {split_cons} 
                        ) as ts1
                        inner join (select site_id, circuit_id, circuit_polarity,  ac_capacity_kw from meta_up23c 
                        ) as m on ts1.circuit_id = m.circuit_id
                        group by site_id, t_stamp,  ac_capacity_kw),
                        data1 as (
                        select site_id, t_stamp, V, P_kW, ac_capacity_kw,
                        lag(t_stamp) over (partition by site_id order by t_stamp) as prev_t_stamp,
                        lag(V) over (partition by site_id order by t_stamp) as prev_V,
                        lag(P_kW) over (partition by site_id order by t_stamp) as prev_P_kW
                        from data
                        ),
                        data2 as (
                        select site_id, t_stamp, V, P_kW, ac_capacity_kw, day(t_stamp) as day,
                        case when (hour(t_stamp) >= 20 or hour(t_stamp) <= 7) then 'day' else 'night' end as day_night,
                        case when P_kW >= 0.04 * ac_capacity_kw then P_kW - 0.04 * ac_capacity_kw else 0 end as nonconformance_sum
                        from data1
                        where V >= 265 and (prev_V >= 265 or prev_P_kW > 0.04 * ac_capacity_kw or P_kW > 0.04 * ac_capacity_kw))
                        select {year} as year, {month} as month, site_id,  day, day_night,
                        sum(nonconformance_sum) as nonconformance_sum, 
                        sum(case when nonconformance_sum > 0 then 1 else 0 end) as nonconformance_count,
                        count(nonconformance_sum) as total_count
                        from data2
                        group by site_id, day, day_night
                        """)
                        # 
    
    print(f"Completed year={year}, month={month}, {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}")
    return df

tasks = [(year, month, split_cons) for year in (2024, 2025) for month in range(1, 13) 
         for split_cons in ['system.bucket(postcode, 16) <= 3', '(system.bucket(postcode, 16) > 3 and system.bucket(postcode, 16) <= 7)', 
                            '(system.bucket(postcode, 16) > 7 and system.bucket(postcode, 16) <= 11)', 'system.bucket(postcode, 16) > 11'] ]
        #  for split_cons in ['system.bucket(postcode, 16) <= 1'] ]
         
df2 = trino_parallel(run_func, tasks, num_workers=num_workers)

Completed year=2024, month=2, (bucket > 3 and bucket <= 7)
Completed year=2024, month=2, bucket > 11
Completed year=2024, month=2, bucket <= 3
Completed year=2024, month=1, bucket > 11
Completed year=2024, month=1, bucket <= 3
Completed year=2024, month=1, (bucket > 3 and bucket <= 7)
Completed year=2024, month=2, (bucket > 7 and bucket <= 11)
Completed year=2024, month=1, (bucket > 7 and bucket <= 11)
Completed year=2024, month=3, bucket <= 3
Completed year=2024, month=3, bucket > 11
Completed year=2024, month=4, (bucket > 7 and bucket <= 11)
Completed year=2024, month=3, (bucket > 7 and bucket <= 11)
Completed year=2024, month=4, bucket > 11
Completed year=2024, month=4, bucket <= 3
Completed year=2024, month=4, (bucket > 3 and bucket <= 7)
Completed year=2024, month=5, bucket <= 3
Completed year=2024, month=3, (bucket > 3 and bucket <= 7)
Completed year=2024, month=5, (bucket > 3 and bucket <= 7)
Completed year=2024, month=5, bucket > 11
Completed year=2024, month=7, bucket <= 3
Com

In [30]:
def run_func(args):
    year, month, split_cons = args
    df = iceberg_sql(f"""
                        with data as (
                            select site_id, t_stamp, sum(cast(power*circuit_polarity as decimal(18, 6)))/1000 as P_kW, 
                                    max(cast(voltage as decimal(18, 6))) as V, ac_capacity_kw
                        from 
                        (select circuit_id, t_stamp, power, energy_reactive, voltage 
                        from ts where year={year} and month={month} and is_pv = true and voltage > 0 and voltage < 300 
                        and {split_cons} 
                        ) as ts1
                        inner join (select site_id, circuit_id, circuit_polarity,  ac_capacity_kw from meta_up23c 
                        ) as m on ts1.circuit_id = m.circuit_id
                        group by site_id, t_stamp,  ac_capacity_kw),
                        data1 as (
                        select site_id, t_stamp, V, P_kW, ac_capacity_kw,
                        lag(t_stamp) over (partition by site_id order by t_stamp) as prev_t_stamp,
                        lag(V) over (partition by site_id order by t_stamp) as prev_V,
                        lag(P_kW) over (partition by site_id order by t_stamp) as prev_P_kW
                        from data
                        ),
                        data2 as (
                        select site_id, t_stamp, V, P_kW, ac_capacity_kw, day(t_stamp) as day,
                        case when (hour(t_stamp) >= 20 or hour(t_stamp) <= 7) then 'day' else 'night' end as day_night,
                        case when P_kW >= 0.04 * ac_capacity_kw then P_kW - 0.04 * ac_capacity_kw else 0 end as nonconformance_sum
                        from data1
                        where V >= 265 )
                        select {year} as year, {month} as month, site_id,  day, day_night,
                        sum(nonconformance_sum) as nonconformance_sum, 
                        sum(case when nonconformance_sum > 0 then 1 else 0 end) as nonconformance_count,
                        count(nonconformance_sum) as total_count
                        from data2
                        group by site_id, day, day_night
                        """)
                        # 
    
    print(f"Completed year={year}, month={month}, {split_cons.replace('system.bucket(postcode, 16)', 'bucket')}")
    return df

tasks = [(year, month, split_cons) for year in (2024, 2025) for month in range(1, 13) 
         for split_cons in ['system.bucket(postcode, 16) <= 3', '(system.bucket(postcode, 16) > 3 and system.bucket(postcode, 16) <= 7)', 
                            '(system.bucket(postcode, 16) > 7 and system.bucket(postcode, 16) <= 11)', 'system.bucket(postcode, 16) > 11'] ]
        #  for split_cons in ['system.bucket(postcode, 16) <= 1'] ]
         
df1 = trino_parallel(run_func, tasks, num_workers=num_workers)

Completed year=2024, month=1, bucket > 11
Completed year=2024, month=1, (bucket > 3 and bucket <= 7)
Completed year=2024, month=2, (bucket > 3 and bucket <= 7)
Completed year=2024, month=2, bucket > 11
Completed year=2024, month=1, bucket <= 3
Completed year=2024, month=2, bucket <= 3
Completed year=2024, month=1, (bucket > 7 and bucket <= 11)
Completed year=2024, month=2, (bucket > 7 and bucket <= 11)
Completed year=2024, month=4, bucket <= 3
Completed year=2024, month=3, bucket <= 3
Completed year=2024, month=4, (bucket > 3 and bucket <= 7)
Completed year=2024, month=3, (bucket > 3 and bucket <= 7)
Completed year=2024, month=4, bucket > 11
Completed year=2024, month=3, bucket > 11
Completed year=2024, month=4, (bucket > 7 and bucket <= 11)
Completed year=2024, month=3, (bucket > 7 and bucket <= 11)
Completed year=2024, month=6, (bucket > 3 and bucket <= 7)
Completed year=2024, month=6, (bucket > 7 and bucket <= 11)
Completed year=2024, month=5, bucket > 11
Completed year=2024, month=

In [32]:
df2.shape, df1.shape, df2['site_id'].nunique(), df1['site_id'].nunique()

((1178, 8), (1347, 8), 198, 234)

In [16]:
df.query("P_kW >= 0.04 * ac_capacity_kw")

,year,month,site_id,t_stamp,t_stamp_next,V,ac_capacity_kw,P_kW,P_kW_next
19,2024,2,1559806445,2024-02-01 01:05:00,2024-02-01 01:10:00,265.20,10.0,8.548817,8.609913
20,2024,2,1559806445,2024-02-01 01:10:00,2024-02-01 01:15:00,266.65,10.0,8.609913,8.620447
21,2024,2,1559806445,2024-02-01 01:15:00,2024-02-01 01:20:00,266.60,10.0,8.620447,8.449297
22,2024,2,1559806445,2024-02-01 01:40:00,2024-02-01 01:45:00,265.90,10.0,8.278790,8.055660
23,2024,2,1559806445,2024-02-01 02:00:00,2024-02-01 02:05:00,266.15,10.0,8.590050,8.460400
...,...,...,...,...,...,...,...,...,...
1,2025,5,615902948,2025-05-03 02:25:00,2025-05-03 02:30:00,265.73,10.0,9.942690,9.871970
2,2025,5,615902948,2025-05-03 02:30:00,2025-05-03 02:35:00,265.05,10.0,9.871970,9.970260
3,2025,5,615902948,2025-05-03 02:35:00,2025-05-03 02:40:00,265.16,10.0,9.970260,9.899350
4,2025,5,615902948,2025-05-03 02:40:00,2025-05-03 02:45:00,265.35,10.0,9.899350,9.967260


In [14]:
df['site_id'].nunique()

234

In [4]:
h

NameError: name 'h' is not defined